# Выполнение ЛР №4: Иерархическая и вероятностная кластеризации

## Подключение библиотек

In [ ]:
import math
import re

import pandas               as pd
import numpy                as np

import matplotlib.pyplot    as plt

from sklearn.preprocessing import StandardScaler

# Импорт библиотек для иерархической кластеризации
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

## Настройка библиотек

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_rows', None)

## Задание 1

### Формулировка

* Изучить датасеты
* Определить, нужно ли проводить 
нормализацию имеющихся данных?
* Выполнить её при необходимости

### Решение

In [ ]:
#### Изучение датасетов

# Загрузка первого датасета (flame.txt)
flame_data = pd.read_csv('Вариант 4/flame.txt', sep='\t', header=None, names=['x', 'y', 'cluster'])
print("Первый датасет (flame.txt):")
print(f"Размер: {flame_data.shape}")
print(f"Столбцы: {flame_data.columns.tolist()}")
print("\nПервые 10 строк:")
display(flame_data.head(10))
print("\nСтатистическое описание:")
display(flame_data.describe())

# Загрузка второго датасета (Seed_Data.csv)
seed_data = pd.read_csv('Вариант 4/Seed/Seed_Data.csv')
print("\n" + "="*50)
print("Второй датасет (Seed_Data.csv):")
print(f"Размер: {seed_data.shape}")
print(f"Столбцы: {seed_data.columns.tolist()}")
print("\nПервые 10 строк:")
display(seed_data.head(10))
print("\nСтатистическое описание:")
display(seed_data.describe())

In [ ]:
#### Анализ необходимости нормализации

# Анализ масштабов данных в первом датасете
print("\n" + "="*50)
print("АНАЛИЗ НЕОБХОДИМОСТИ НОРМАЛИЗАЦИИ")
print("\nПервый датасет (flame.txt):")
print(f"Диапазон X: {flame_data['x'].min():.2f} - {flame_data['x'].max():.2f}")
print(f"Диапазон Y: {flame_data['y'].min():.2f} - {flame_data['y'].max():.2f}")
print(f"Стандартное отклонение X: {flame_data['x'].std():.2f}")
print(f"Стандартное отклонение Y: {flame_data['y'].std():.2f}")

# Анализ масштабов данных во втором датасете
print("\nВторой датасет (Seed_Data.csv):")
features = seed_data.drop('target', axis=1)
for col in features.columns:
    print(f"{col}: диапазон {features[col].min():.3f} - {features[col].max():.3f}, std = {features[col].std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# График первого датасета
axes[0].scatter(flame_data['x'], flame_data['y'], c=flame_data['cluster'])
axes[0].set_title('Первый датасет (flame.txt) - исходные данные')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# График второго датасета (первые два признака)
scatter = axes[1].scatter(seed_data['A'], seed_data['C'], c=seed_data['target'], alpha=0.7)
axes[1].set_title('Второй датасет (Seed) - Area vs Perimeter')
axes[1].set_xlabel('Area (A)')
axes[1].set_ylabel('Compactness (C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#### Выполнение нормализации

# Для первого датасета
print("\n" + "="*50)
print("НОРМАЛИЗАЦИЯ ДАННЫХ")

# Первый датасет - координаты имеют схожие масштабы, но для кластеризации лучше нормализовать
flame_features = flame_data[['x', 'y']].copy()
scaler_flame = StandardScaler()
flame_normalized = scaler_flame.fit_transform(flame_features)
flame_data_norm = pd.DataFrame(flame_normalized, columns=['x_norm', 'y_norm'])
flame_data_norm['cluster'] = flame_data['cluster']

print("Первый датасет после нормализации:")
display(flame_data_norm.describe())

# Второй датасет - признаки имеют разные масштабы, нормализация необходима
seed_features = seed_data.drop('target', axis=1)
scaler_seed = StandardScaler()
seed_normalized = scaler_seed.fit_transform(seed_features)
seed_data_norm = pd.DataFrame(seed_normalized, columns=[f'{col}_norm' for col in seed_features.columns])
seed_data_norm['target'] = seed_data['target']

print("\nВторой датасет после нормализации:")
display(seed_data_norm.describe())

In [ ]:
# Визуализация нормализованных данных
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# График первого датасета после нормализации
axes[0].scatter(flame_data_norm['x_norm'], flame_data_norm['y_norm'], 
               c=flame_data_norm['cluster'])
axes[0].set_title('Первый датасет - нормализованные данные')
axes[0].set_xlabel('X (нормализованный)')
axes[0].set_ylabel('Y (нормализованный)')
axes[0].grid(True, alpha=0.3)

# График второго датасета после нормализации
axes[1].scatter(seed_data_norm['A_norm'], seed_data_norm['C_norm'], 
               c=seed_data_norm['target'], alpha=0.7)
axes[1].set_title('Второй датасет - нормализованные данные')
axes[1].set_xlabel('Area (нормализованная)')
axes[1].set_ylabel('Compactness (нормализованный)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Вывод

1. Первый датасет (flame.txt):
   - Содержит 240 точек с координатами (x, y) и метками кластеров
   - Координаты имеют схожие масштабы, но нормализация улучшит качество кластеризации
   - Данные представляют собой двумерные точки, образующие характерную форму

2. Второй датасет (Seed_Data.csv):
   - Содержит 210 образцов зерен пшеницы с 7 геометрическими признаками
   - Признаки имеют существенно разные масштабы (от 0.87 до 16.63)
   - Нормализация РЕКОМЕНДУЕТСЯ для корректной работы алгоритмов кластеризации
   - Данные содержат 3 класса (сорта пшеницы)

3. Нормализация выполнена методом StandardScaler для обоих датасетов
   - Все признаки приведены к стандартному нормальному распределению (μ=0, σ=1)
   - Это обеспечит равный вклад всех признаков в процесс кластеризации

## Задание 2 (Датасет #1)

### Формулировка

Для первого датасета:
* Определить число кластеров, построив дендрограмму.

### Решение

#### Построение дендрограмм с разными методами связывания

In [ ]:
flame_features_norm = flame_data_norm[["x_norm", "y_norm"]]
methods = ['ward', 'complete', 'average', 'single']
metrics = ['euclidean', 'manhattan', 'chebyshev', 'hamming']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

linkage_matrices = {}

for i, method in enumerate(methods):
    # Вычисление матрицы связей
    linkage_matrix = linkage(flame_features_norm, method=method, metric=metrics[0])
    
    linkage_matrices[method] = linkage_matrix
    
    # Построение дендрограммы
    dendrogram(linkage_matrix, ax=axes[i], truncate_mode='lastp', p=20, leaf_font_size=8)
    axes[i].set_title(f'Дендрограмма (метод: {method})')
    axes[i].set_xlabel('Индекс кластера или размер кластера')
    axes[i].set_ylabel('Расстояние')

plt.tight_layout()
plt.show()

#### Детальная дендрограмма для метода Ward (наиболее подходящий для данной задачи)

In [ ]:
plt.figure(figsize=(20, 8))

# Построение полной дендрограммы
ward_linkage = linkage_matrices['ward']
dendrogram(ward_linkage, leaf_rotation=90, leaf_font_size=5)
plt.title('Полная дендрограмма (метод Ward)', fontsize=14)
plt.xlabel('Индекс образца')
plt.ylabel('Расстояние')

plt.show()

#### Добавляет линии порогов для 2х и 3х кластеров

In [ ]:
ward_linkage = linkage_matrices['ward']
plt.figure(figsize=(20, 8))
dendrogram(ward_linkage, leaf_rotation=90, leaf_font_size=5)
plt.title('Полная дендрограмма (метод Ward)', fontsize=14)
plt.xlabel('Индекс образца')
plt.ylabel('Расстояние')
plt.axhline(y=16, color='red', linestyle='--', alpha=0.7, label='Порог для 2 кластеров')
plt.axhline(y=13, color='orange', linestyle='--', alpha=0.7, label='Порог для 3 кластеров')

plt.legend()
plt.show()

#### Выводы

1. АНАЛИЗ ДЕНДРОГРАММЫ:
    - Наибольшие скачки на последних этапах слияния можно наблюдать в методах `Complete` и `Ward`
    - Метод `Ward` показал наилучшие результаты для данного датасета
    - Дендрограмма показывает два или три основных кластера

## Задание 3 (Датасет #1)

### Формулировка

Для первого датасета:
* Построить  график  зависимости  расстояний  между кластерами от шага слияния. 
* Определить по графику оптимальное число кластеров.

### Решение

In [ ]:
# Извлекаем расстояния слияния из матрицы связей
ward_linkage = linkage_matrices['ward']
distances = ward_linkage[:, 2]

#### График зависимости расстояний от шага слияния

In [ ]:
# Создаем массив шагов слияния (от 1 до n-1, где n - количество точек)
steps = np.arange(1, len(distances) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Полный график всех шагов слияния
axes[0].plot(steps, distances, 'b-', linewidth=2, marker='o', markersize=3)
axes[0].set_title('Расстояния слияния на всех шагах', fontsize=12)
axes[0].set_xlabel('Шаг слияния')
axes[0].set_ylabel('Расстояние между кластерами')
axes[0].grid(True, alpha=0.3)

# График 2: Последние 30 шагов (наиболее важные для определения числа кластеров)
last_steps = 30
axes[1].plot(steps[-last_steps:], distances[-last_steps:], 'r-', linewidth=2, marker='o', markersize=5)
axes[1].set_title(f'Последние {last_steps} шагов слияния', fontsize=12)
axes[1].set_xlabel('Шаг слияния')
axes[1].set_ylabel('Расстояние между кластерами')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Графики приращений расстояний

In [ ]:
# Вычисляем разницу между последовательными расстояниями
distance_diffs = np.diff(distances)

# График приращений
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Все приращения
axes[0].plot(range(2, len(distances) + 1), distance_diffs, 'g-', linewidth=2, marker='o', markersize=3)
axes[0].set_title('Приращения расстояний между шагами', fontsize=12)
axes[0].set_xlabel('Шаг слияния')
axes[0].set_ylabel('Приращение расстояния')
axes[0].grid(True, alpha=0.3)

# График 2: Последние приращения (в обратном порядке - от большего к меньшему числу кластеров)
last_steps_diff = 15
x_clusters = np.arange(last_steps_diff, 0, -1)
y_diffs = distance_diffs[-last_steps_diff:]

axes[1].plot(x_clusters, y_diffs, 'r-', linewidth=2, marker='o', markersize=6)
axes[1].set_title('Приращения расстояний (последние шаги)', fontsize=12)
axes[1].set_xlabel('Количество кластеров')
axes[1].set_ylabel('Приращение расстояния')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis()  # Инвертируем ось X для удобства чтения

# Выделяем наибольшие скачки
max_diff_idx = np.argmax(distance_diffs[-last_steps_diff:])
axes[1].axvline(x=x_clusters[max_diff_idx], color='orange', linestyle='--', 
                alpha=0.7, label=f'Максимальный скачок ({x_clusters[max_diff_idx]} кластеров)')
axes[1].legend()

plt.tight_layout()
plt.show()

#### Определение оптимального числа кластеров методом "локтя"

In [ ]:
# Визуализация оптимального числа кластеров
plt.figure(figsize=(14, 6))

# Строим график в обратном порядке (от многих кластеров к малому числу)
n_points = 25
clusters_range = np.arange(n_points, 0, -1)
distances_subset = distances[-n_points:]

plt.plot(clusters_range, distances_subset, 'b-', linewidth=2.5, marker='o', markersize=7)
plt.title('Определение оптимального числа кластеров (метод "локтя")', fontsize=14)
plt.xlabel('Количество кластеров', fontsize=12)
plt.ylabel('Расстояние слияния', fontsize=12)
plt.grid(True, alpha=0.3)

# Отмечаем потенциальные оптимальные точки
optimal_candidates = [2, 3, 4]
colors = ['red', 'orange', 'green']
for k, color in zip(optimal_candidates, colors):
    idx = n_points - k
    plt.scatter(k, distances_subset[idx], s=200, c=color, alpha=0.8, 
                edgecolors='black', linewidth=2, zorder=5,
                label=f'{k} кластера')

plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()

### Вывод

**Анализ графиков:**

1. **График расстояний слияния** показывает, что наибольшие скачки происходят на последних этапах слияния, когда количество кластеров уменьшается с 4 до 3 и с 2 до 1.

2. **График приращений расстояний** четко демонстрирует "локти" - точки, где приращение резко возрастает:
   - Самый большой скачок наблюдается при переходе от 2 к 1 кластеру
   - Второй значительный скачок - при переходе от 4 к 3 кластерам

3. **Оптимальное число кластеров:**
   - **4 кластера** - наиболее очевидный выбор, так как это на следующем этапе слияния наблюдается первый существенный скачок.
   - **2 кластера** - также хороший вариант, учитывая что он по величине приращения является наибольшим. Стоит учитывать, что слияние от 3 кластеров к 2 не дало значительного приращения расстояния, что свидетельствует о формировании неестественной группы. 
   
4. **Рекомендация:** Для датасета flame, на основе графика зависимости расстояния между кластерами от шага слияния, оптимальным является **4 кластера**. Окончательный выбор будет сделан в следующем задании после визуализации результатов кластеризации.

## Задание 4 (Датасет #1) <a id="mb_z4">

### Формулировка

Для первого датасета:
* Выбрать  оптимальный  алгоритм  и  выполнить 
кластеризацию.  
* Пробовать  разные  варианты,  чтобы определить наиболее подходящий вариант. 

### Решение

## Задание 5 (Датасет #2)

### Формулировка

Для прикладного датасета (второй)
* Определить число кластеров 
    * методом расчёта сумм расстояний от точек данных до центра  ближайшего  к  ней  кластера  (инерций).  
* Для кластеризации используйте как исходный датасет без  предобработки,  так  и  с  предобработкой, выполненной в лабораторной работе 3. 

### Решение

## Задание 6 (Датасет #2) <a id="mb_z6">

### Формулировка

Для прикладного датасета (второй)
*  Выполнить  кластеризацию  методом  k-средних.

### Решение

## Задание 7 (Датасет #1,2)

### Формулировка

* Провести анализ полученных результатов из заданий [4](#mb_z4) и 
[6](#mb_z6)
* Cформулировать отличительные особенности разных 
кластеров для прикладного датасета.

### Решение